# Assignment 2.3 MolGlyph Recognition Module
**Yihang Xu | AAI6630**

This notebook implements Stage 3 of the group pipeline: molecule crop images → decoded SMILES. It reads an upstream `crops.csv`, runs the pretrained MolGlyph checkpoint once per crop, and writes `raw_predictions.csv` for the RDKit validation module. The upstream detector and downstream structure evaluation are separate modules.

## 1. Paths and input contract

Keep this notebook in your `AAI6630 Benchmarking Code` folder, alongside `BioMiner-main`, `molglyph_large.pt`, and `outputs`. First run your 1.2 notebook through the XN crop generation cell. The next cell creates a four-crop demonstration manifest from its saved XN images. If Nishtha provides `crops.csv`, set `CROPS_CSV` to that file and skip Section 2.

In [3]:
from pathlib import Path
from time import perf_counter
import importlib.util
import sys

import pandas as pd
import torch

PROJECT_DIR = Path.cwd()
BIOMINER_ROOT = PROJECT_DIR / "BioMiner-main"
MODEL_PATH = PROJECT_DIR / "molglyph_large.pt"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "phase2_recognition"
CROPS_CSV = OUTPUT_DIR / "crops.csv"
PREDICTIONS_CSV = OUTPUT_DIR / "raw_predictions.csv"
DEVICE = "cpu"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", PROJECT_DIR)
print("Checkpoint found:", MODEL_PATH.is_file())
print("BioMiner found:", BIOMINER_ROOT.is_dir())

Working directory: C:\Users\yxu\Desktop\AAI6630 Benchmarking Code
Checkpoint found: True
BioMiner found: True


## 2. Demonstration input from the 1.2 XN crops

Run this cell only to make a local demonstration input. The images were manually cropped from paper 2, page 2 in Assignment 1.2; these are **not** MolDetv2 detections. `crop_path` is relative to the manifest directory. Coordinates are intentionally omitted because this recognition stage does not use them.

In [ ]:
XN_IMAGE_DIR = PROJECT_DIR / "outputs" / "opencv_pilot" / "benchmark" / "xn_images"
samples = [
    ("2_p2_001", "paper_2_sample_01.png", "AKI-062a"),
    ("2_p2_002", "paper_2_sample_02.png", "D4476"),
    ("2_p2_003", "paper_2_sample_03.png", "FP1-24"),
    ("2_p2_004", "paper_2_sample_04.png", "Compound 1h"),
]
missing = [name for _, name, _ in samples if not (XN_IMAGE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"XN crops missing from {XN_IMAGE_DIR}: {missing}. "
        "Run the XN crop cell in your 1.2 notebook first."
    )

demo = pd.DataFrame([
    {"crop_id": crop_id, "paper_id": "2", "page": 2,
     "crop_path": (Path("..") / "opencv_pilot" / "benchmark" / "xn_images" / name).as_posix(),
     "molecule_label": label}
    for crop_id, name, label in samples
])
demo.to_csv(CROPS_CSV, index=False)
display(demo)

## 3. Load MolGlyph and the official decoder

This reuses the model loading and abbreviation reconstruction method from the successful 1.2 notebook. MolGlyph is pretrained; no fine-tuning or data augmentation is performed. The model's internal image preparation is handled by BioMiner. Crops are passed as saved without additional resizing or normalization in this notebook.

In [7]:
OCSR_PACKAGE_ROOT = BIOMINER_ROOT / "BioMiner" / "MolScribe"
utility_path = BIOMINER_ROOT / "BioMiner" / "commons" / "molglyph_utils.py"
if not (OCSR_PACKAGE_ROOT / "molscribe").is_dir():
    raise FileNotFoundError(f"MolGlyph package missing: {OCSR_PACKAGE_ROOT}")
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"MolGlyph checkpoint missing: {MODEL_PATH}")
if not utility_path.is_file() or not (utility_path.parent / "abbrevs.csv").is_file():
    raise FileNotFoundError("BioMiner molglyph_utils.py and abbrevs.csv are required")

package_path = str(OCSR_PACKAGE_ROOT.resolve())
if package_path not in sys.path:
    sys.path.insert(0, package_path)
from molscribe import MolGlyph

ocsr_model = MolGlyph(str(MODEL_PATH), device=torch.device(DEVICE), num_workers=1)
spec = importlib.util.spec_from_file_location("molglyph_postprocessing", utility_path)
molglyph_utils = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = molglyph_utils
spec.loader.exec_module(molglyph_utils)

def decode_caption(raw_caption):
    """Expand a MolGlyph caption with BioMiner's official abbreviation decoder."""
    caption = raw_caption.strip()
    if not caption or "<sep>" not in caption:
        return caption
    translated = molglyph_utils.Translator.refactor(caption)
    if translated is None or not isinstance(translated.smi, str):
        raise ValueError("The official decoder could not reconstruct the caption")
    return translated.smi

print("MolGlyph loaded on", DEVICE)

D:\Anaconda\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
D:\Anaconda\Lib\site-packages\onmt\modules\sparse_activations.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
D:\Anaconda\Lib\site-packages\onmt\modules\sparse_activations.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
D:\Anaconda\Lib\site-packages\onmt\modules\sparse_losses.py:12: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
D:\Anaconda\Lib\site-packag

MolGlyph loaded on cpu


## 4. Read crops, run inference, and save the handoff CSV

The required input fields are `crop_id`, `paper_id`, and `crop_path`. One output row is kept per input even if prediction fails. `raw_caption` preserves the model response; `raw_smiles` is its decoded SMILES. RDKit validity, wildcard status, and canonicalization belong to Sanika's next stage.

In [9]:
def load_crop_manifest(path):
    """Check the Stage 2 handoff and resolve image paths from the CSV folder."""
    crops = pd.read_csv(path, dtype=str, keep_default_na=False)
    required = {"crop_id", "paper_id", "crop_path"}
    missing_columns = required - set(crops.columns)
    if missing_columns:
        raise ValueError(f"Missing crops.csv columns: {sorted(missing_columns)}")
    if crops[list(required)].eq("").any().any() or crops["crop_id"].duplicated().any():
        raise ValueError("crop_id must be unique; required fields cannot be empty")
    return crops

def recognize_crops(crops, manifest_path):
    """Run one inference per image, retaining status and elapsed time."""
    records = []
    for _, sample in crops.iterrows():
        image_path = Path(sample["crop_path"])
        if not image_path.is_absolute():
            image_path = manifest_path.parent / image_path
        record = {"crop_id": sample["crop_id"], "paper_id": sample["paper_id"],
                  "raw_smiles": "", "inference_time": 0.0,
                  "raw_caption": "", "recognition_status": "ok", "error": ""}
        start = perf_counter()
        try:
            if not image_path.is_file():
                raise FileNotFoundError(f"Image missing: {image_path}")
            caption = ocsr_model.predict_image_file(
                str(image_path), return_atoms_bonds=False, return_confidence=False
            )
            if not isinstance(caption, str):
                raise TypeError(f"Expected a string, received {type(caption).__name__}")
            record["raw_caption"] = caption
            record["raw_smiles"] = decode_caption(caption)
            if not record["raw_smiles"]:
                record["recognition_status"] = "empty_output"
        except Exception as error:
            record["recognition_status"] = "recognition_error"
            record["error"] = f"{type(error).__name__}: {error}"
        record["inference_time"] = round(perf_counter() - start, 6)
        records.append(record)
    return pd.DataFrame(records, columns=[
        "crop_id", "paper_id", "raw_smiles", "inference_time",
        "raw_caption", "recognition_status", "error"
    ])

crops = load_crop_manifest(CROPS_CSV)
predictions = recognize_crops(crops, CROPS_CSV)
predictions.to_csv(PREDICTIONS_CSV, index=False)
display(predictions[["crop_id", "paper_id", "raw_smiles", "inference_time", "recognition_status", "error"]])
print("Saved:", PREDICTIONS_CSV)

,crop_id,paper_id,raw_smiles,inference_time,recognition_status,error
0,2_p2_001,2,CN(C)CC(=O)Nc1n[nH]c2ccc(-c3cn(Cc4ccccc4)nn3)cc12,1.244501,ok,
1,2_p2_002,2,NC(=O)c1ccc(-c2nc(-c3ccc4c(c3)OCCO4)c(-c3ccccn...,1.388750,ok,
2,2_p2_003,2,COc1cnc(COc2ccc(F)c(F)c2)cc1-c1cc2c([nH]1)CCNC2=O,1.229535,ok,
3,2_p2_004,2,CC(C)(O)c1ccc(Nc2nc3ccc(C#N)cc3[nH]2)cc1,1.208364,ok,


Saved: C:\Users\yxu\Desktop\AAI6630 Benchmarking Code\outputs\phase2_recognition\raw_predictions.csv


## 5. Check the saved result

This cell verifies the handoff structure. Inspect the displayed SMILES and error statuses before committing the CSV. The four XN crops are a small module smoke test, not document-level accuracy evidence.

In [11]:
saved = pd.read_csv(PREDICTIONS_CSV, dtype=str, keep_default_na=False)
assert len(saved) == len(crops)
assert saved["crop_id"].tolist() == crops["crop_id"].tolist()
assert {"crop_id", "paper_id", "raw_smiles", "inference_time"}.issubset(saved.columns)
print("Rows:", len(saved))
print("Statuses:", saved["recognition_status"].value_counts().to_dict())
display(saved[["crop_id", "raw_smiles", "recognition_status", "error"]])

Rows: 4
Statuses: {'ok': 4}


,crop_id,raw_smiles,recognition_status,error
0,2_p2_001,CN(C)CC(=O)Nc1n[nH]c2ccc(-c3cn(Cc4ccccc4)nn3)cc12,ok,
1,2_p2_002,NC(=O)c1ccc(-c2nc(-c3ccc4c(c3)OCCO4)c(-c3ccccn...,ok,
2,2_p2_003,COc1cnc(COc2ccc(F)c(F)c2)cc1-c1cc2c([nH]1)CCNC2=O,ok,
3,2_p2_004,CC(C)(O)c1ccc(Nc2nc3ccc(C#N)cc3[nH]2)cc1,ok,


## Integration notes

The detector supplies images and a `crops.csv` with unique `crop_id`, `paper_id`, and `crop_path`. Set `CROPS_CSV` to its actual manifest, skip Section 2, and rerun Sections 3–5. The output columns match the group 1.4 handoff, with additional caption, status, and error fields for traceability. Confirm with the validation lead whether `raw_smiles` should contain decoded captions as implemented here. Source code and decoder come from BioMiner; this notebook adapts the author's Assignment 1.2 inference workflow to a group-stage CSV interface. The checkpoint and course images are external dependencies, not embedded in this file.